In [19]:
import pandas as pd
import numpy as np





# Con este dataframe vamos a generar un plano interactivo del recorrido de las diferentes lineas de subte,donde se podra ver la accesibilidad de cada Estacion. Esto es la cantidad de Ascensores y Escaleras Mecanicas que hay en cada Estacion de cada una de las lineas de Subte. 

In [20]:
# CARGA DEL DATASET Y VISUALIZACIÓN DE LOS DATOS

longitudLineas=pd.read_csv("Longitud_Lineas.csv")
longitudLineas= longitudLineas.sort_values(by='lineasub')
longitudLineas.head(10)


,lineasub,longitud,longitud_km
0,LINEA A,0.103193,11.487418
1,LINEA B,0.124921,13.906191
2,LINEA C,0.041123,4.577821
3,LINEA D,0.107143,11.927151
4,LINEA E,0.122758,13.665399
5,LINEA H,0.075421,8.395814


In [21]:
#RENOMBRO  A COLUNA 'lineasub' por 'linea'.

longitudLineas = longitudLineas.rename(columns={'lineasub': 'linea'})

In [22]:
#QUITO LA PALABRA 'LINEA' DE LA COLUMNA 'linea' para que quede solo la letra de la Linea de Subterraneo.

longitudLineas['linea'] = longitudLineas['linea'].str.replace('LINEA ', '')

In [23]:
longitudLineas.head(10)

,linea,longitud,longitud_km
0,A,0.103193,11.487418
1,B,0.124921,13.906191
2,C,0.041123,4.577821
3,D,0.107143,11.927151
4,E,0.122758,13.665399
5,H,0.075421,8.395814


In [24]:
#CARGO UN SEGUNDO DATASET CON LAS ESTACIONES DE SUBTE ACCESIBLES (Ascensores y Escaleras Mecanicas por Estacion por Linea)

df=pd.read_csv("estaciones-accesibles.csv")
df= df.sort_values(by='linea')
df.head(10)


,long,lat,linea,estacion,escaleras_mecanicas,ascensores
0,-58.436429,-34.618280,A,ACOYTE,2,2
14,-58.469640,-34.630707,A,SAN PEDRITO,3,2
13,-58.463541,-34.629087,A,SAN JOSE DE FLORES,4,3
12,-58.386777,-34.609413,A,SAENZ PEÑA,2,0
10,-58.441178,-34.620405,A,PRIMERA JUNTA,3,2
9,-58.406707,-34.609817,A,PLAZA MISERERE,6,0
8,-58.370968,-34.608810,A,PLAZA DE MAYO,0,1
11,-58.448648,-34.623529,A,PUAN,3,3
6,-58.374268,-34.608559,A,PERU,2,2
5,-58.415186,-34.610782,A,LORIA,1,1


**LOS DATOS DE 'long' y 'lat', sirven para ubicar que accesor hay en cada estacion en un plano.**

In [25]:
#AGREGO UNA NUEVA FEATURE O COLUMNA CON EL NOMBRE "lomgitud_km", EL CUAL VA A TENER LA LONGTUD DE CADA LINEA DE SUBTERRANEO.
df['longitud_km'] = df['linea'].map(longitudLineas.set_index('linea')['longitud_km'])

**APROVECHO ESA INFORMACION PARA CREAR UN PLANO INTERACTIVO CON LAS ESTACIONES DE SUBTE Y LA ACCESIBILIDAD EN CADA UNA DE ELLAS , POR LINEA DE SUBTE.**

- Para ello utilizamos la libreria "folium"

In [26]:
import folium
import pandas as pd


## El siguiente codigo, toma las coordenadas del Dataset y crea el plano en un archivo HTML.

**Explicacion del siguiente codigo.**

- Este script crea un mapa interactivo de accesibilidad del subte de Buenos Aires utilizando la librería Folium.

- A continuación, se describe su funcionamiento:

	1.	Inicialización del mapa:
- Se centra el mapa en las coordenadas de la Ciudad de Buenos Aires (location=[-34.6037, -58.3816]) con un nivel de zoom adecuado (zoom_start=12).

	2.	Función de color según accesibilidad:
- La función color_accesibilidad() asigna un color a cada estación:
	•	Verde: estación totalmente accesible (tiene escaleras mecánicas y ascensores).
	•	Naranja: estación parcialmente accesible (tiene uno de los dos).
	•	Rojo: estación no accesible (no tiene ninguno).

	3.	Marcadores por estación:
- Se recorren las filas del DataFrame df, y para cada estación se agrega un círculo de color sobre su ubicación geográfica, con un popup informativo que muestra el nombre de la estación y la cantidad de escaleras y ascensores.

	4.	Leyenda visual:
- Se añade una leyenda fija en el mapa para interpretar los colores utilizados en los marcadores.

	5.	Exportación del mapa:
- El mapa se guarda como un archivo HTML llamado mapa_accesibilidad_subte_bsas.html, que puede abrirse en cualquier navegador.

In [27]:
# Creo un mapa centrado en Buenos Aires
mapa = folium.Map(location=[-34.6037, -58.3816], zoom_start=12)

# Función para determinar el color del marcador según la accesibilidad
def color_accesibilidad(escaleras, ascensores):
    if escaleras > 0 and ascensores > 0:
        return 'green'  # Estación totalmente accesible
    elif escaleras > 0 or ascensores > 0:
        return 'orange' # Estación parcialmente accesible
    else:
        return 'red'    # Estación no accesible

# Agrego marcadores para cada estación.
for index, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=5,
        color=color_accesibilidad(row['escaleras_mecanicas'], row['ascensores']),
        fill=True,
        fill_color=color_accesibilidad(row['escaleras_mecanicas'], row['ascensores']),
        fill_opacity=0.7,
        popup=f"<b>{row['estacion']}</b><br>Escaleras mecánicas: {row['escaleras_mecanicas']}<br>Ascensores: {row['ascensores']}"
    ).add_to(mapa)

# Agrego leyenda
legend_html = """
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 250px; height: 120px; 
            border:2px solid grey; z-index:9999; padding:10px; background-color:white;">
    <p><b>Leyenda de accesibilidad</b></p>
    <p><i class="fa fa-circle" style="color:green"></i> Estación totalmente accesible</p>
    <p><i class="fa fa-circle" style="color:orange"></i> Estación parcialmente accesible</p>
    <p><i class="fa fa-circle" style="color:red"></i> Estación no accesible</p>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legend_html))

# Guardar el mapa como un archivo HTML
mapa.save("mapa_accesibilidad_subte_bsas.html")

print("Mapa de accesibilidad generado y guardado como mapa_accesibilidad_subte_bsas.html")


Mapa de accesibilidad generado y guardado como mapa_accesibilidad_subte_bsas.html
